# Run all pipelines

End-to-end driver for the Formulation A study. Runs, in order:

1. **Experiments** - EV/EEVS (population + sample mean), WS, DE via `run_experiments.run_all()`.
2. **Jensen gap** - ex-post Formulation B evaluation from the DE CSV (no re-solve).
3. **Figures** - executes `plotter.ipynb` to regenerate the CV / Jensen-gap map PDFs.
4. **Warm-start benchmark** (optional) - runtime-only comparison, writes `_warmstart`-suffixed files.

All outputs land under `results/` (per-scenario Gurobi logs go to `results/solver_logs/`).

> Stage 1 is slow (hundreds of MIQCP solves, can run for hours). Stages 2-3 are fast.
> Each stage is its own cell so you can re-run just the part you need.

In [ ]:
import sys
import time
from pathlib import Path

SRC_DIR = str(Path.cwd().parent / 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Force a fresh import of the model modules so edits in ../src take effect
# without needing a kernel restart.
import importlib
for _m in ['paths', 'gurobi_utils', 'ev_eevs_model', 'ws_model', 'de_model',
           'run_experiments', 'jensen_gap_from_de_csv', 'benchmark_warm_start']:
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

import run_experiments
print('src on path:', SRC_DIR)

## Stage 1 - Experiments (EV / EEVS / WS / DE)

Writes the per-model CSVs, `..._summary.json`, the `.txt` summary and
`scarce_<scarce>_solve_diagnostics.md` into `results/`, plus one Gurobi log per
solve into `results/solver_logs/`. **This is the long one.**

In [ ]:
_t0 = time.time()
results_by_scarcity = run_experiments.run_all()
print(f'\n[stage 1] experiments finished in {time.time() - _t0:.1f}s')

## Stage 2 - Jensen gap (ex-post Formulation B)

Reads the DE CSV written by stage 1 and evaluates Formulation B's
`log(E[phi**alpha])` objective on that same allocation. No Gurobi solve.

`OF_A_DE_check` should match `OF_DE` in `..._summary.json`; if it doesn't, the
DE CSV and the summary are out of sync (i.e. stale analysis).

In [ ]:
import jensen_gap_from_de_csv as jg
from svi_data import SVI

csv_path = jg.find_latest_de_csv()
print(f'Reading DE solution from: {csv_path}')
jensen_result = jg.jensen_gap_from_de_csv(csv_path, svi=SVI)

print(f"Scenarios: {jensen_result['num_scenarios']}")
print(f"OF_A (DE, recomputed check): {jensen_result['OF_A_DE_check']:.6f}")
print(f"OF_B (ex-post):              {jensen_result['OF_B_expost']:.6f}")
print(f"Jensen gap:                  {jensen_result['jensen_gap']:.6f}")

### Freshness check

Confirms the recomputed DE objective matches the summary JSON, so the Jensen-gap
numbers really describe the run you just did (this is the exact staleness bug
caught previously - see `docs/dossier.md` Section 14.4).

In [ ]:
import glob, json as _json
from paths import results_path

_summaries = sorted(glob.glob(results_path('scarce_*_summary.json')))
if _summaries:
    with open(_summaries[-1]) as f:
        _s = _json.load(f)
    _diff = abs(_s['OF_DE'] - jensen_result['OF_A_DE_check'])
    print(f"summary.json OF_DE      : {_s['OF_DE']:.6f}")
    print(f"recomputed OF_A_DE_check: {jensen_result['OF_A_DE_check']:.6f}")
    print(f"abs diff                : {_diff:.2e}")
    print('OK - analysis is fresh' if _diff < 1e-6 else 'STALE - regenerate the Jensen gap against the current DE CSV')
else:
    print('No summary JSON found - run stage 1 first.')

## Stage 3 - Figures

Executes `plotter.ipynb` in place, regenerating the dispersion-CV and Jensen-gap
map PDFs in `results/` against the data produced above.

In [ ]:
import subprocess

_t0 = time.time()
_proc = subprocess.run(
    [sys.executable, '-m', 'nbconvert', '--to', 'notebook', '--execute', '--inplace',
     'plotter.ipynb', '--ExecutePreprocessor.timeout=600'],
    cwd=str(Path.cwd()), capture_output=True, text=True)
print(_proc.stdout[-2000:])
print(_proc.stderr[-2000:])
print(f'[stage 3] plotter exit code {_proc.returncode} in {time.time() - _t0:.1f}s')

In [ ]:
from paths import RESULTS_DIR

for _p in sorted(Path(RESULTS_DIR).glob('*_map.pdf')):
    print(f'{_p.name:40s} {_p.stat().st_size:>9,d} bytes  {time.ctime(_p.stat().st_mtime)}')

## Stage 4 - Warm-start benchmark (optional)

Runtime-only experiment: re-runs EEVS (sample mean) and WS with sequential MIP
warm starts. Results are written to `_warmstart`-suffixed files so they can
**never** overwrite the baseline outputs from stage 1.

Compare `results/scarce_<scarce>_solve_diagnostics_warmstart.md` against
`results/scarce_<scarce>_solve_diagnostics.md` for the speedup.

In [ ]:
from benchmark_warm_start import run_benchmark

_t0 = time.time()
run_benchmark(0.75)
print(f'\n[stage 4] warm-start benchmark finished in {time.time() - _t0:.1f}s')

### Warm-start vs baseline runtime

Parses both diagnostics files and reports total Gurobi runtime per model.

In [ ]:
import re
from collections import defaultdict
from paths import results_path

def _totals(md_path):
    tot = defaultdict(float)
    if not Path(md_path).exists():
        return tot
    for line in Path(md_path).read_text().splitlines():
        if not line.startswith('|'):
            continue
        cols = [c.strip() for c in line.strip('|').split('|')]
        if len(cols) < 2:
            continue
        label = cols[0]
        for c in cols:
            if re.fullmatch(r'\d+\.\d+', c):
                model = label.split('_')[0]
                tot[model] += float(c)
                break
    return tot

_base = _totals(results_path('scarce_0.75_solve_diagnostics.md'))
_warm = _totals(results_path('scarce_0.75_solve_diagnostics_warmstart.md'))

for m in sorted(set(_base) | set(_warm)):
    b, w = _base.get(m, 0.0), _warm.get(m, 0.0)
    if b and w:
        print(f'{m:8s} baseline {b:9.1f}s   warmstart {w:9.1f}s   {(b - w) / b * 100:+6.1f}%')
    else:
        print(f'{m:8s} baseline {b:9.1f}s   warmstart {w:9.1f}s')